In [19]:
%load_ext autoreload
%autoreload 2

import open3d as o3d
import numpy as np
import os
import sys
from glob import glob
import pickle
import json
from pathlib import Path
from copy import deepcopy

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')
from utils.write_output import write_array2txt

from cylinder_extraction.cylinder import Cylinder
from cylinder_extraction.station import Station
from registration.utils import multiway_load_point_clouds as load_point_clouds
from registration.utils import multiway_full_registration_with_sequence as full_registration_with_sequence
from registration.utils import multiway_full_registration_with_pairs as full_registration_with_pairs
from registration.utils import pairwise_registration, combine_point_cloud, multiway_filter_point_clouds

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Input
The first part of the tutorial code reads three point clouds from files. The point clouds are downsampled and visualized together. They are misaligned.

In [20]:
root_dir = Path('D:/REGISTRATION DATA/External/JB-531 40ft EXT')
data_dir = root_dir / 'target_tank' / 'global' / 'ground'

In [21]:
with open(os.path.join(root_dir, 'globalization_axis_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)
all_cylinders = bundle.target_tank

In [22]:
# with open(root_dir / 'tank_info.pkl', 'rb') as f:
#     tank_info = pickle.load(f)
# all_cylinders = tank_info['tanks'][tank_label]


In [23]:
in_cyls, out_cyls = multiway_filter_point_clouds(all_cylinders, radius_offset_max=0.5)

In [24]:
out_cyls

[]

### Read point clouds for inlier cylinders

In [25]:
# with open(os.path.join(root_dir, 'thin_slice', 'station_order.txt'), 'r') as f:
#     order = json.load(f)
# order = order['order']
# print(order)

# read pairs
with open(root_dir / 'target_tank' / 'global' / 'thin_slice' / 'station_pairs.txt', 'r') as f:
    pair_info = json.load(f)
pair_info = pair_info['pair']
# pair_info = [[p0-4, p1-4] for p0,p1 in pair_info]
print(pair_info)
# order = range(len(in_cyls))

[[3, 5], [0, 1], [6, 5], [2, 5], [0, 7], [1, 4], [4, 2]]


In [26]:
all_pcds_down, all_pcds_full = load_point_clouds(in_cyls, file_dir=data_dir, voxel_size=0.01)

D:\REGISTRATION DATA\External\JB-531 40ft EXT\target_tank\global\ground\Citgo Tank 4- Setup 004 cluster_12.pcd
D:\REGISTRATION DATA\External\JB-531 40ft EXT\target_tank\global\ground\Citgo Tank 4- Setup 005 cluster_7.pcd
D:\REGISTRATION DATA\External\JB-531 40ft EXT\target_tank\global\ground\Citgo Tank 4- Setup 007 cluster_9.pcd
D:\REGISTRATION DATA\External\JB-531 40ft EXT\target_tank\global\ground\Citgo Tank 4- Setup 008 cluster_11.pcd
D:\REGISTRATION DATA\External\JB-531 40ft EXT\target_tank\global\ground\Tank4 Scan_Citgo_Tank_4-_Setup_006 cluster_9.pcd
D:\REGISTRATION DATA\External\JB-531 40ft EXT\target_tank\global\ground\Tank4 Scan_Citgo_Tank_4-_Setup_009 cluster_3.pcd
D:\REGISTRATION DATA\External\JB-531 40ft EXT\target_tank\global\ground\Tank4 Scan_Citgo_Tank_4-_Setup_010 cluster_4.pcd
D:\REGISTRATION DATA\External\JB-531 40ft EXT\target_tank\global\ground\Tank4 Scan_Citgo_Tank_4-_Setup_011 cluster_7.pcd


In [27]:
pcds_down = all_pcds_down
pcds_full = all_pcds_full

## Pose graph

In [28]:
print("Full registration ...")
max_correspondence_distance_coarse = 0.2#voxel_size * 15
max_correspondence_distance_fine1  = 0.01#voxel_size * 1.5
max_correspondence_distance_fine2  = 0.005#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration_with_pairs(
        pcds_down, pair_info,
        max_correspondence_distance_coarse,
        max_correspondence_distance_fine1,
        max_correspondence_distance_fine2
    )

Full registration ...
----SOURCE, TARGET = 3, 5
Coarse alignment
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.9516, RMSE 0.0335
[Open3D DEBUG] Residual : 1.72e-04 (# of elements : 2069454)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.9518, RMSE 0.0320
[Open3D DEBUG] Residual : 6.91e-05 (# of elements : 2069918)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.9518, RMSE 0.0319
[Open3D DEBUG] Residual : 6.87e-05 (# of elements : 2069912)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.9518, RMSE 0.0319
[Open3D DEBUG] Residual : 6.89e-05 (# of elements : 2069929)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.9518, RMSE 0.0319
[Open3D DEBUG] Residual : 6.89e-05 (# of elements : 2069926)
FINE ALIGNMENT PHASE I
[Open3D DEBUG] ICP Iteration #0: Fitness 0.8049, RMSE 0.0045
[Open3D DEBUG] Residual : 1.17e-06 (# of elements : 1750418)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.8046, RMSE 0.0045
[Open3D DEBUG] Residual : 9.04e-07 (# of elements : 1749723)
[Open3D DEBUG] ICP Iteration #2: 

Open3D uses the function `global_optimization` to perform pose graph optimization. Two types of optimization methods can be chosen: `GlobalOptimizationGaussNewton` or `GlobalOptimizationLevenbergMarquardt`. The latter is recommended since it has better convergence property. Class `GlobalOptimizationConvergenceCriteria` can be used to set the maximum number of iterations and various optimization parameters.

Class `GlobalOptimizationOption` defines a couple of options. `max_correspondence_distance` decides the correspondence threshold. `edge_prune_threshold` is a threshold for pruning outlier edges. `reference_node` is the node id that is considered to be the global space.

In [29]:
# max_correspondence_distance_fine = 0.005
print("Optimizing PoseGraph ...")
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine2,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)

Optimizing PoseGraph ...
[Open3D DEBUG] Validating PoseGraph - finished.
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 8 nodes and 7 edges.
[Open3D DEBUG] Line process weight : 21.645986
[Open3D DEBUG] [Initial     ] residual : 1.154173e+05, lambda : 2.744520e+03
[Open3D DEBUG] [Iteration 00] residual : 4.890174e+00, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] [Iteration 01] residual : 4.405940e-03, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] [Iteration 02] residual : 2.304377e-06, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] [Iteration 03] residual : 6.600724e-10, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] Current_residual < 1.000000e-06
[Open3D DEBUG] [GlobalOptimizationLM] total time : 0.000 sec.
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 8 nodes and 7 edges.
[Open3D DEBUG] Line process weight : 21.645986
[Open3D DEBUG] [Initial     ] residual : 6.600724e-10, lambda : 2.758628e+03
[Open3D DEBUG] Delta.norm() < 1.0000

The global optimization performs twice on the pose graph. The first pass optimizes poses for the original pose graph taking all edges into account and does its best to distinguish false alignments among uncertain edges. These false alignments have small line process weights, and they are pruned after the first pass. The second pass runs without them and produces a tight global alignment. In this example, all the edges are considered as true alignments, hence the second pass terminates immediately.

## Visualize optimization
The transformed point clouds are listed and visualized using `draw_geometries`.

In [12]:
# print("Transform points and display")
# for point_id in range(len(pcds_down)):
#     pcds_down[point_id].transform(pose_graph.nodes[point_id].pose)
# o3d.visualization.draw_geometries(pcds_down,
#                                   zoom=0.3412,
#                                   front=[0.4257, -0.2125, -0.8795],
#                                   lookat=[2.6172, 2.0475, 1.532],
#                                   up=[-0.0694, -0.9768, 0.2024])

## Make a combined point cloud
`PointCloud` has a convenience operator `+` that can merge two point clouds into a single one. In the code below, the points are uniformly resampled using `voxel_down_sample` after merging. This is recommended post-processing after merging point clouds since it can relieve duplicated or over-densified points.

In [13]:
inliner_pcd_combined_down = combine_point_cloud(deepcopy(pcds_down), pose_graph)
write_array2txt(
    os.path.join(data_dir, f'inliers_combined_with_ids_down.txt'),
    inliner_pcd_combined_down)


In [14]:
# write a slice
l = 3.0
thickness = 0.01
ind = ((l <= inliner_pcd_combined_down[:,2]) & (inliner_pcd_combined_down[:,2] <= l + thickness))
write_array2txt(
    os.path.join(data_dir, f'slice_3_down_optimize_.txt'),
    inliner_pcd_combined_down[ind,:]
)

In [30]:
inliner_pcd_combined = combine_point_cloud(pcds_full, pose_graph)
# pcds = load_point_clouds(path_,voxel_size)
# pcd_combined = o3d.geometry.PointCloud()
# pcd_full = np.empty(shape=[0,4])
# for point_id in range(len(pcds)):
#     pcds[point_id].transform(pose_graph.nodes[point_id].pose)
#     A = np.asarray(pcds[point_id].points)
#     B = np.c_[ A, np.ones( A.shape[0] )*point_id ]
#     pcd_full = np.r_[pcd_full,B]

In [15]:
pcd_full = np.empty(shape=[0,4])
for point_id in range(len(all_pcds_down)):

    A = np.asarray(all_pcds_down[point_id].points)
    B = np.c_[ A, np.ones( A.shape[0] )*point_id ]
    pcd_full = np.r_[pcd_full,B]


In [31]:
write_array2txt(
    os.path.join(data_dir, f'inliner_pcds_combined_with_ids.txt'),
    inliner_pcd_combined)


In [70]:
o3d.io.write_point_cloud(
    os.path.join(data_dir, f'all_down.pcd'),
    pcd, print_progress=True
)

NameError: name 'pcd' is not defined

## FINISH

In [ ]:
inliner_pcd_combined.shape

(642487958, 4)

In [ ]:
# write a slice
l = 3.0
thickness = 0.01
ind = ((l <= inliner_pcd_combined[:,2]) & (inliner_pcd_combined[:,2] <= l + thickness))
write_array2txt(
    os.path.join(data_dir, f'slice_inliers_combined_with_ids.txt'),
    inliner_pcd_combined[ind,:]
)

In [63]:
ind = inliner_pcd_combined[:,-1]==5
inliner_pcd_combined[ind,2].min()

-0.002986272441479922

In [56]:
inliner_pcd_combined[ind,-1].max()

7.0

In [32]:
write_array2txt(
    os.path.join(data_dir, f'tank_{tank_name}_without004_no_ids.txt'),
    inliner_pcd_combined[:,:3])

In [17]:
p8 = pcds_down[8]
p9 = pcds_down[9]

In [21]:
p7 = pcds_down[7]

In [23]:
min(p9.compute_point_cloud_distance(p7))

0.0005837807807712218

In [18]:
dist89 = p8.compute_point_cloud_distance(p9)

In [20]:
min(dist89)

0.5343259772835293

## Handle outlier cylinders

In [34]:
cyl_combined = Cylinder(
    metadata = {
        'ptc_filepath': os.path.join(data_dir, f'tank_{tank_name}_without004_no_ids.txt'), #'inliers_combined.txt'),
        'tank_id': 1,
        'station': Station(name=f'tank_{tank_name}_without004_no_ids')
    }
)
out_cyls.insert(0,cyl_combined)

In [41]:
# read points
pcds_down, pcds_full = load_point_clouds(out_cyls)
# pose graph
print("Full registration ...")
max_correspondence_distance_coarse = 1#voxel_size * 15
max_correspondence_distance_fine = 0.01#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration(pcds_down,
                                   max_correspondence_distance_coarse,
                                   max_correspondence_distance_fine)
# optimize
print("Optimizing PoseGraph ...")
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)
# make combined
pose_graph.nodes.pop(0)
outlier_pcd_combined = combine_point_cloud(pcds_full[1:], pose_graph)
# inlier + outlier
pcd_combined = np.r_[inliner_pcd_combined, outlier_pcd_combined]
# pcd_combined = combine_point_cloud(pcds_full, pose_graph)
# write
write_array2txt(
    data_dir + '\complete_combined_with_ids.txt',
    pcd_combined)

D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\tank_1\tank_1_without004_no_ids.txt
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\tank_1\004.txt
Full registration ...
----SOURCE 0----
===== source_id=0, target_id=1 =====
Apply point-to-plane ICP
Coarse alignment
[Open3D DEBUG] ICP Iteration #0: Fitness 0.3063, RMSE 0.2806
[Open3D DEBUG] Residual : 3.11e-02 (# of elements : 1849886)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.3055, RMSE 0.2698
[Open3D DEBUG] Residual : 2.67e-02 (# of elements : 1845158)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.3056, RMSE 0.2698
[Open3D DEBUG] Residual : 2.75e-02 (# of elements : 1845523)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.3056, RMSE 0.2703
[Open3D DEBUG] Residual : 2.80e-02 (# of elements : 1845495)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.3055, RMSE 0.2703
[Open3D DEBUG] Residual : 2.81e-02 (# of elements : 1845070)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.3056, RMSE 0.2699
[Open3D DEBUG] Residual : 2.84e-02 (# of elements : 